# clueboard

> The clue board (Phase 11): each subtask's answer is fixed by modular clues ("x mod p = r") dealt to different agents, so no agent can solve it alone; open, submit, share and wait in the Phase 5 world protocol, with share and solve logs, net value per ledger row, the pooled-information bound, and individual, team and credit rewards.

In [ ]:
#| default_exp clueboard

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass

import numpy as np

from explRL.cluetask import ClueTask, makeClueTask
from explRL.multi import spawnRngs


@dataclass
class Subtask:
    id: int
    value: float
    task: ClueTask
    appear: float
    expire: float
    holders: list                       # per clue j: frozenset of agents holding it
    solvedBy: int = -1


class ClueBoardEnv:
    """The clue board (Phase 11) in the Phase 5 world protocol, time in turns: as the Phase 10 board, but each
    subtask's answer is fixed by m modular clues dealt to different agents. Actions: ('open', k), ('submit', x),
    ('share', (k, j, r)) (records the share of clue j of open subtask k with residue r), ('wait', None)."""
    timeUnit = 'turn'

    def __init__(self, seed, nAgents, m=3, r=1, R=105, value=(1, 2, 3), life=30, penalty=0.5, scheme='individual',
                 credit=0.5):
        self.m, self.r, self.R, self.values, self.life = int(m), int(r), int(R), tuple(value), float(life)
        self.penalty, self.scheme, self.credit = float(penalty), scheme, float(credit)
        self.reset(seed, nAgents)

    def reset(self, seed, nAgents):
        self.rng, self.rngDeal = spawnRngs(seed, 2)
        N = self.nAgents = int(nAgents)
        self.tWorld, self.serial, self.tasks, self.open, self.ledger, self.netRows = 0.0, 0, {}, [], [], []
        self.opened, self.last, self.pending, self.wrongBy = [None] * N, [0.0] * N, np.zeros(N), np.zeros(N, int)
        self.shareLog, self.solveLog, self.submits, self.bound, self._noted, self._tBound = [], [], [], [], {}, None
        self.counts = dict(solved=0, wrong=0, late=0, opens=0, shares=0)
        M = 2 * N + 2
        for k in range(M):                 # staggered ages, so expiries spread over the first `life` turns
            self._new(-self.life * k / M)

    def _deal(self):
        "Holders per clue: r agents each, least-loaded first, never completing an agent's set when avoidable."
        N, held, out = self.nAgents, np.zeros(self.nAgents, int), []
        for _ in range(self.m):
            hs = set()
            for _ in range(min(self.r, N)):
                cand = [a for a in range(N) if a not in hs]
                safe = [a for a in cand if held[a] < self.m - 1] or cand
                lo = min(held[a] for a in safe)
                pool = [a for a in safe if held[a] == lo]
                a = int(pool[int(self.rngDeal.integers(len(pool)))])
                hs.add(a)
                held[a] += 1
            out.append(frozenset(hs))
        return out

    def _new(self, t):
        v = float(self.values[int(self.rng.integers(len(self.values)))])
        self.tasks[self.serial] = Subtask(self.serial, v, makeClueTask(self.rng, self.m, self.R), t, t + self.life,
                                          self._deal())
        self.open.append(self.serial)
        self.serial += 1

    def advance(self, t):
        if t < self.tWorld:
            raise ValueError('time went backwards')
        while due := [k for k in self.open if self.tasks[k].expire <= t]:
            k = min(due, key=lambda j: (self.tasks[j].expire, j))
            self.open.remove(k)
            self._new(self.tasks[k].expire)
        self.tWorld = float(t)
        if self._tBound != t:              # the pooled bound, once per turn: top-N open values, two turns per solve
            self._tBound = t
            self.bound.append(sum(sorted((self.tasks[k].value for k in self.open), reverse=True)[:self.nAgents]) / 2)

    def held(self, i, k):
        "Clues agent i holds of subtask k: {j: r}."
        s = self.tasks[k]
        return {j: s.task.clue(j)[1] for j, hs in enumerate(s.holders) if i in hs}

    def holders(self, k):
        return list(self.tasks[k].holders)

    def noteSubmit(self, i, nCandidates):
        "The submitter's number of candidates before its next submission (reported by the agent or adaptor)."
        self._noted[i] = int(nCandidates)

    def _creditTo(self, i, k):
        "Agents whose true shares first supplied the clues of k that solver i did not hold."
        out = []
        for j, hs in enumerate(self.tasks[k].holders):
            if i not in hs:
                first = next((s[1] for s in self.shareLog if s[2] == k and s[3] == j and not s[6] and s[1] != i), None)
                if first is not None and first not in out:
                    out.append(first)
        return out

    def _reward(self, i, delta, k=None, solved=False):
        if self.scheme == 'team':
            self.pending += delta / self.nAgents
        elif self.scheme == 'credit' and solved and (to := self._creditTo(i, k)):
            for a in to:
                self.pending[a] += self.credit * delta / len(to)
            self.pending[i] += (1 - self.credit) * delta
        else:                              # 'individual', and credit with nothing to credit
            self.pending[i] += delta
        r, self.pending[i] = float(self.pending[i]), 0.0
        return r

    def apply(self, i, action, t):
        kind, arg = action
        self.advance(t)
        k, h, delta, solved = self.opened[i], 0.0, 0.0, False
        if kind == 'open' and arg in self.open:
            self.opened[i], self.counts['opens'] = int(arg), self.counts['opens'] + 1
            row, res, loc = (i, t, 1.0, 'travel', int(arg), None, 0.0), 'opened', int(arg)
        elif kind == 'submit' and k is not None:
            s, loc = self.tasks[k], k
            if not s.task.check(arg):
                delta, res = -self.penalty, 'wrong'
                self.counts['wrong'] += 1
                self.wrongBy[i] += 1
            elif k in self.open:
                h = delta = s.value
                s.solvedBy, res, solved = i, 'solved', True
                self.solveLog.append((float(t), i, k, not all(i in hs for hs in s.holders)))
                self.counts['solved'] += 1
                self.open.remove(k)
                self._new(t)
            else:
                res = 'late'
                self.counts['late'] += 1
            self.submits.append((float(t), i, k, self._noted.pop(i, None), res))
            row = (i, t, 1.0, 'stay', k, k if solved else None, h)
        elif kind == 'share' and arg[0] in self.open:
            k2, j, r = int(arg[0]), int(arg[1]), int(arg[2])
            redundant = any(s[2] == k2 and s[3] == j for s in self.shareLog)
            self.shareLog.append((float(t), i, k2, j, r, redundant, self.tasks[k2].task.clue(j)[1] != r))
            self.counts['shares'] += 1
            row, res, loc = (i, t, 1.0, 'stay', k2, None, 0.0), 'shared', k2
        else:
            row, res, loc = (i, t, 1.0, 'stay', -1, None, 0.0), 'wait' if kind == 'wait' else 'invalid', -1
        rew = self._reward(i, delta, k, solved)
        self.ledger.append(row + (rew, float(t) if row[3] == 'stay' else float('nan'),
                                  float(t) + 1.0 if row[3] == 'stay' else float('nan')))
        self.netRows.append(delta)
        self.last[i] = rew
        info = dict(kind=row[3], harvest=h, nHere=1 if row[3] == 'stay' else 0, patch=row[5], dur=1.0, task=loc,
                    result=res, net=delta)
        return rew, 1.0, loc, info

    def observe(self, i):
        return dict(time=self.tWorld, opened=self.opened[i], lastReward=self.last[i],
                    open=[(k, self.tasks[k].value, self.tasks[k].expire) for k in self.open])

    def pooledBound(self):
        "Mean over the run's turns of the team value per turn with every clue known to all."
        return float(np.mean(self.bound)) if self.bound else 0.0

    def netByAgent(self):
        "Value solved minus penalties paid, per agent (welfare; independent of the reward scheme)."
        v = np.zeros(self.nAgents)
        for s in self.tasks.values():
            if s.solvedBy >= 0:
                v[s.solvedBy] += s.value
        return v - self.penalty * self.wrongBy

## Tests

In [ ]:
import numpy as np
from explRL.cluetask import CLUE_PAIRS
from explRL.clueboard import *

# AC-3 dealing: no agent holds all clues whenever r m <= N (m - 1); a solo agent holds all; r holders per clue
for m, R in CLUE_PAIRS:
    for r in (1, 2):
        for N in (1, 2, 3, 4, 8):
            b = ClueBoardEnv(7, N, m=m, r=r, R=R)
            for s in b.tasks.values():
                assert all(len(hs) == min(r, N) for hs in s.holders)
                full = [i for i in range(N) if all(i in hs for hs in s.holders)]
                if N == 1:
                    assert full == [0]
                elif r * m <= N * (m - 1):
                    assert not full, (m, r, N)

# scoring: open costs a turn; only the first correct answer scores; wrong answers pay; shares are logged
b = ClueBoardEnv(1, 4, m=3, r=1, R=105, life=100, penalty=0.5)
k = b.open[0]; s = b.tasks[k]
r, dt, loc, info = b.apply(0, ('open', k), 0.0)
assert (r, dt, loc, info['kind'], info['result']) == (0.0, 1.0, k, 'travel', 'opened')
b.apply(1, ('open', k), 0.0)
p, res = s.task.clue(1)
b.noteSubmit(0, 7)
assert b.apply(0, ('submit', (s.task.x + 1) % 105), 1.0)[0] == -0.5
b.apply(2, ('share', (k, 1, res)), 1.0); b.apply(3, ('share', (k, 1, res)), 2.0); b.apply(3, ('share', (k, 2, (s.task.clue(2)[1] + 1) % s.task.clue(2)[0])), 3.0)
assert [(x[1], x[5], x[6]) for x in b.shareLog] == [(2, False, False), (3, True, False), (3, False, True)]
r, _, _, info = b.apply(0, ('submit', s.task.x), 4.0)
assert r == s.value and info['result'] == 'solved' and k not in b.open
assert b.apply(1, ('submit', s.task.x), 4.0)[3]['result'] == 'late'
assert b.solveLog == [(4.0, 0, k, not all(0 in hs for hs in s.holders))] and b.submits[0][3] == 7
assert b.counts == dict(solved=1, wrong=1, late=1, opens=2, shares=3)
assert b.apply(2, ('share', (999, 0, 0)), 5.0)[3]['result'] == 'invalid' and b.apply(2, ('wait', None), 6.0)[3]['result'] == 'wait'
assert len(b.ledger) == len(b.netRows) and np.isclose(sum(b.netRows), b.netByAgent().sum()) and b.pooledBound() > 0
assert all(len(row) == 10 for row in b.ledger) and b.held(0, b.open[0]) == {j: b.tasks[b.open[0]].task.clue(j)[1] for j, hs in enumerate(b.tasks[b.open[0]].holders) if 0 in hs}

# rewards: individual, team and credit give the same team total (delivered plus pending)
def play(scheme):
    e = ClueBoardEnv(2, 3, m=3, r=1, R=105, life=100, scheme=scheme)
    k = e.open[0]; s = e.tasks[k]; solver = 0
    missing = [j for j, hs in enumerate(s.holders) if solver not in hs]
    t, rews = 0.0, []
    rews.append(e.apply(solver, ('open', k), t)[0]); t += 1
    for j in missing:
        sharer = next(iter(s.holders[j]))
        rews.append(e.apply(sharer, ('share', (k, j, s.task.clue(j)[1])), t)[0]); t += 1
    rews.append(e.apply(solver, ('submit', (s.task.x + 1) % 105), t)[0]); t += 1
    rews.append(e.apply(solver, ('submit', s.task.x), t)[0]); t += 1
    return sum(rews) + e.pending.sum(), e, rews, missing
tot = {sc: play(sc)[0] for sc in ('individual', 'team', 'credit')}
assert np.isclose(tot['individual'], tot['team']) and np.isclose(tot['individual'], tot['credit'])
_, e, rews, missing = play('credit')
v = e.tasks[e.solveLog[0][2]].value
assert missing and np.isclose(rews[-1], 0.5 * v) and np.isclose(e.pending.sum(), 0.5 * v)   # sharers get c v, pending

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()